## Functions

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import sys, os
from ipywidgets import interact
from matplotlib.gridspec import GridSpec

In [2]:
def pib(x, l, n):
  box = (x > 0)
  box *= (x < l)
  return (2/l)**.5 * np.sin(n * np.pi * x / l)*box

def pibE(l, mass, n):
  h = 6.626e-34
  return n**2 * h**2 / (8*mass*l**2)

def pib2D(lmax, lx, ly, nx, ny,pixels=300):
  x = np.linspace(0, lmax, pixels)
  y = np.linspace(0, lmax, pixels)
  X, Y = np.meshgrid(x, y)
  psi = pib(X, lx, nx) * pib(Y, ly, ny)
  return X, Y, psi

def lvl_pairs(n_max):
  sums = []
  pairs_ = []
  for n in range(1, n_max + 1):
    for m in range(1,n_max+1):
      sums.append(n+m)
      pairs_.append([n,m])
  pairs_ = np.array(pairs_)
  order = list(np.argsort(sums))
  return pairs_[order]

def energy_dict(lx, ly, mass, n_max):
  energy_dict = {}
  pairs = lvl_pairs(n_max)
  E_mag = [n**2/lx**2 + m**2/ly**2 for n,m in pairs ]
  order = np.argsort(E_mag)
  for n,m in pairs[order]:
    name = f'({n},{m})'
    energy_dict[name] = {}
    energy_dict[name]['E'] = pibE(lx, mass, n) + pibE(ly, mass, m)
    energy_dict[name]['nx'] = n
    energy_dict[name]['ny'] = m

  return energy_dict



In [ ]:

# Comparing two levels
def pib_interact(n_max):
  ns = [n for n in range(1, n_max+1)]
  @interact(
      nx = widgets.Dropdown(options=ns),
      ny = widgets.Dropdown(options=ns),
      lx=widgets.FloatSlider(
          value=1.0,
          min=0.5,
          max=2.0,
          step=0.05,
          description='Lx (nm):'
      ),
      ly=widgets.FloatSlider(
          value=1.0,
          min=0.5,
          max=2.0,
          step=0.05,
          description='Ly (nm):'
      )
  )
  def g(nx, ny, lx,ly):
    me = 9.101e-31
    fig, ax = plt.subplots(
        ncols=2,
        figsize=(11, 4),
        gridspec_kw={"width_ratios": [1.3, 1.0]}
    )
    fig.subplots_adjust(wspace=0.4)
    lx_m = lx*1e-9
    lmax = 2e-9
    x_pixel = 300
    ly_m = ly*1e-9

    X, Y, psi = pib2D(lmax, lx_m, ly_m, nx, ny)
    zmax = np.max(psi)
    zmin = -zmax
    im0 = ax[0].imshow(psi, cmap='seismic', extent=( 0,lmax/1e-9,0, lmax/1e-9),
                aspect='equal',
                vmin = zmin,
                vmax = zmax,
                origin='lower'
                )
    ax[0].set_xlabel('x (nm)')
    ax[0].set_ylabel('y (nm)')
    ax[0].set_title(f"$n_x$={nx}, $n_y$={ny}")
    cbar = fig.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

    e_dict = energy_dict(lx_m, ly_m, me, n_max)
    i=0
    ax[1].set_title('(nx,ny)')
    for e in e_dict:
      if i%2 ==0:
        ha='right'
      else:
          ha='left'
      i+=1
      ax[1].plot(0, e_dict[e]['E'], 'r_' , markersize='50')
      ax[1].text(0, e_dict[e]['E'], e, ha=ha)
    ax[1].plot(0, pibE(lx_m, me, nx) + pibE(ly_m, me, ny), 'b_', markersize='50')
    ax[1].set_ylabel("E (J)")



In [ ]:

# Mixing two levels
def pib_mix(nmax):
  lx_m = 1e-9
  ly_m = 1e-9
  me = 9.101e-31
  e_dict = energy_dict(lx_m, ly_m, me, nmax)
  levels = [e for e in e_dict]

  lmax = 1e-9

  @interact(
        state1=widgets.Dropdown(options=levels,
                                value=levels[3],
                                description='ψ1: (nx, ny)'),
        state2=widgets.Dropdown(options=levels,
                                value=levels[4],
                                description='ψ2: (nx, ny)'),
        theta=widgets.FloatSlider(
            value=0,
            min=0,
            max=np.pi/2,
            step=0.05,
            description='θ'))
  def g(state1, state2, theta):
    nx1 = e_dict[state1]['nx']
    ny1 = e_dict[state1]['ny']
    nx2 = e_dict[state2]['nx']
    ny2 = e_dict[state2]['ny']
    c1 = np.cos(theta)
    c2 = np.sin(theta)

    fig, ax = plt.subplots(ncols=3, figsize=(10, 5))
    fig.subplots_adjust(wspace=0.4)

    X, Y, psi1 = pib2D(lmax, lx_m, ly_m, nx1, ny1)
    X, Y, psi2 = pib2D(lmax, lx_m, ly_m, nx2, ny2)
    psi = c1*psi1 + c2*psi2
    vmax = np.max(psi1)
    vmin = -vmax

    ax[0].imshow(psi1, cmap='seismic',
                vmin=vmin, vmax=vmax,
                extent=(0, 1, 0, 1))
    ax[0].set_title(f'$\\psi_1$: nx={nx1}, ny={ny1}')

    ax[1].imshow(psi2, cmap='seismic',
                vmin=vmin, vmax=vmax,
                extent=(0, 1, 0, 1))
    ax[1].set_title(f'$\\psi_2$: nx={nx2}, ny={ny2}')
    ax[2].imshow(psi, cmap='seismic',vmin=vmin, vmax=vmax,
                extent=(0, 1, 0, 1))
    ax[2].set_title(
        f'$\\psi$: {c1:.2f}$\\psi_1$ + {c2:.2f}$\\psi_2$')
    E1 = e_dict[state1]['E']
    E2 = e_dict[state2]['E']
    print(f" Expected value <E> = {c1**2*E1 + c2**2*E2:.2e} J")

In [55]:
#Slice
def pib_slice(n_max):
  ns = [n for n in range(1, n_max+1)]
  @interact(
      nx = widgets.Dropdown(options=ns, value=3),
      ny = widgets.Dropdown(options=ns, value=4),
      ys=widgets.FloatSlider(
          value=0.25,
          min=0.05,
          max=0.95,
          step=0.05,
          description='y slice (nm):'
      ),
      xs=widgets.FloatSlider(
          value=.25,
          min=0.05,
          max=0.95,
          step=0.05,
          description='x slice (nm):'
      )
  )
  def g(nx, ny, xs, ys):
    me = 9.101e-31
    lx =1 
    ly =1

    fig = plt.figure(figsize=(5,5))
    gs = GridSpec(2, 2, figure=fig, hspace=.4, wspace=.6)
    ax0 = fig.add_subplot(gs[0, 1], projection='3d')
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[0,0])
    ax3 = fig.add_subplot(gs[1,1])
    lx_m = lx*1e-9
    lmax = 1e-9
    ly_m = ly*1e-9

    pixels = 300
    X, Y, psi = pib2D(lmax, lx_m, ly_m, nx, ny,pixels=pixels)
    #psi /= 1e9
    zmax = np.max(psi)
    zmin = -zmax

    x = np.linspace(0, lx, pixels)
    y = np.linspace(0, ly, pixels)
    y_row = int(xs/lx * pixels)
    x_row = int(ys/ly * pixels)

    surf = ax0.plot_surface(X*1e9, Y*1e9, psi/1e9,
                             antialiased=True,
                               cmap='seismic',
                                 vmin=zmin/1e9,
                                   vmax=zmax/1e9)
    ax0.plot_wireframe(X*1e9, Y*1e9, psi/1e9,
                        rstride=15,
                          cstride=15,
                          color='k',
                            linewidth=.5)
    ax0.set_xlabel("x (nm)")
    ax0.set_ylabel("y (nm)")
    ax0.set_zlabel("$\\psi(x,y)$")


    ax0.set_zlim(-zmax*2/1e9, zmax*2/1e9)

    ax1.imshow(psi, cmap='seismic', vmin=-zmax, vmax=zmax, extent=(0, lx, 0, 1), origin='lower')
    ax1.axhline(ys, color='k')
    ax1.axvline(xs, color='k')
    ax1.set_xlabel("x (nm)")
    ax1.set_ylabel("y (nm)")
    ax2.plot(x, psi[x_row])
    ax2.set_xlabel('x (nm)')
    ax2.set_ylabel(f'$\\psi(x,{ys:.2f} nm)$ \n(nm$^{{-1}}$)')
    ax2.set_ylim(zmin*1.1, zmax*1.1)
    ax3.plot(psi[:, y_row],y)
    ax3.set_xlim(zmin*1.1, zmax*1.1)
    ax3.set_ylabel("y (nm)")
    ax3.set_xlabel(f"$\\psi({xs:.2f}nm, y)$ \n(nm$^{{-1}}$)")



## 2D Particle-in-a-box

$\psi(x,y) = \dfrac{2}{\sqrt{L_x L_y}}\sin\left(\dfrac{n_x\pi}{L_x}x\right)\sin\left(\dfrac{n_y\pi}{L_y}y\right)$

In [56]:
pib_slice(5)

interactive(children=(Dropdown(description='nx', index=2, options=(1, 2, 3, 4, 5), value=3), Dropdown(descript…


# Degeneracy and  Energy levels
$E_{n_x,n_y} = \dfrac{h^2 }{8 m}\left(\dfrac{n_x^2}{L_x^2} + \dfrac{n_y^2}{L_y^2}\right)$

In [ ]:
pib_interact(n_max=4)


interactive(children=(Dropdown(description='nx', options=(1, 2, 3, 4), value=1), Dropdown(description='ny', op…

## Mixing two 2D levels

In [6]:
pib_mix(3)

interactive(children=(Dropdown(description='ψ1: (nx, ny)', index=1, options=('(1,1)', '(1,2)', '(2,1)', '(2,2)…